In [4]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Machine Learning/Housing.csv'
housing = pd.DataFrame(pd.read_csv(file_path))

features = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

X = torch.tensor(housing[features].values, dtype=torch.float32)
y = torch.tensor(housing['price'].values, dtype=torch.float32)

torch.manual_seed(42)
n = X.shape[0]
n_val = int(0.2 * n)
idx = torch.randperm(n)
train_idx, val_idx = idx[:-n_val], idx[-n_val:]
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

X_mean, X_std = X_train.mean(0), X_train.std(0)
y_mean, y_std = y_train.mean(), y_train.std()

X_train_n = (X_train - X_mean) / X_std
X_val_n   = (X_val   - X_mean) / X_std
y_train_n = (y_train - y_mean) / y_std
y_val_n   = (y_val   - y_mean) / y_std
def loss_fn(t_p, t_c):
    return ((t_p - t_c)**2).mean()
def model_multi(X, params):
    return X @ params[:-1] + params[-1]

def r2_score(y_pred, y_true):
    return 1 - ((y_true - y_pred)**2).sum() / ((y_true - y_true.mean())**2).sum()

def training_loop_val(n_epochs, optimizer, params, X_tr, y_tr, X_v, y_v):
    for epoch in range(1, n_epochs + 1):
        train_loss = loss_fn(model_multi(X_tr, params), y_tr)
        with torch.no_grad():
            val_pred = model_multi(X_v, params)
            val_loss = loss_fn(val_pred, y_v)
            val_r2 = r2_score(val_pred, y_v)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Train loss {train_loss.item():.4f}, "
                  f"Val loss {val_loss.item():.4f}, Val R² {val_r2.item():.4f}")
    return params, val_loss.item()

results2 = {}
for lr in [1e-1, 1e-2, 1e-3, 1e-4]:
    print(f"\n Learning rate {lr}")
    params = torch.zeros(X_train_n.shape[1] + 1, requires_grad=True)
    opt = optim.SGD([params], lr=lr)
    params, vl = training_loop_val(5000, opt, params, X_train_n, y_train_n, X_val_n, y_val_n)
    results2[lr] = (params.detach().clone(), vl)
    print("Params (w1..w5, b):", params.detach())

best_lr2 = min(results2, key=lambda lr: results2[lr][1])
print(f"\nBest lr: {best_lr2}, val loss: {results2[best_lr2][1]:.4f}")

housing3 = housing.copy()
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for c in binary_cols:
    housing3[c] = housing3[c].map({'yes': 1, 'no': 0})

housing3 = pd.get_dummies(housing3, columns=['furnishingstatus'], drop_first=True, dtype=float)

features3 = [c for c in housing3.columns if c != 'price']
print(features3, len(features3))

X3 = torch.tensor(housing3[features3].values, dtype=torch.float32)

X3_train, X3_val = X3[train_idx], X3[val_idx]
X3_mean, X3_std = X3_train.mean(0), X3_train.std(0)
X3_train_n = (X3_train - X3_mean) / X3_std
X3_val_n   = (X3_val   - X3_mean) / X3_std

results3 = {}
for lr in [1e-1, 1e-2, 1e-3, 1e-4]:
    print(f"\n Learning rate {lr}")
    params = torch.zeros(X3_train_n.shape[1] + 1, requires_grad=True)
    opt = optim.SGD([params], lr=lr)
    params, vl = training_loop_val(5000, opt, params, X3_train_n, y_train_n, X3_val_n, y_val_n)
    results3[lr] = (params.detach().clone(), vl)
    print("Params:", params.detach())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Learning rate 0.1
Epoch 500, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 1000, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 1500, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 2000, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 2500, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 3000, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 3500, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 4000, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 4500, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Epoch 5000, Train loss 0.4335, Val loss 0.3617, Val R² 0.5298
Params (w1..w5, b): tensor([ 3.9422e-01,  2.9652e-02,  3.0098e-01,  2.4814e-01,  1.7493e-01,
        -8.8662e-08])

 Learning rate 0.01
Epoch 500, Train loss 0.4335, Val loss 0.3616, Val R² 0.5299
Epoch 1000, Train loss 0.4335, Val 